<a href="https://colab.research.google.com/github/beatrizzzzz/Estudo-com-o-PERMA-Profiler/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Caso seja necessário instalar as bibliotecas:
# !pip install pandas matplotlib scikit-learn pyreadstat

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA


# ============================================================
# 1. LEITURA DO BANCO
# ============================================================

INPUT_SAV = "Banco_Liana_com_Escores(1).sav"
OUTPUT_IMAGE = "grafico_pca_clusters_perma.png"

df = pd.read_spss(INPUT_SAV)


# ============================================================
# 2. SELEÇÃO DAS DIMENSÕES DO PERMA
# ============================================================

perma_cols = [
    "Escore_P",
    "Escore_E",
    "Escore_R",
    "Escore_M",
    "Escore_A",
]

# Verificar se todas as colunas existem no banco
missing_cols = [col for col in perma_cols if col not in df.columns]

if missing_cols:
    raise ValueError(
        f"As seguintes colunas não foram encontradas no banco: {missing_cols}"
    )

# Remover casos com dados ausentes nas dimensões analisadas
df_pca = df.dropna(subset=perma_cols + ["Escore_Geral"]).copy()


# ============================================================
# 3. PADRONIZAÇÃO DOS ESCORES
# ============================================================

scaler = StandardScaler()

X = scaler.fit_transform(df_pca[perma_cols])


# ============================================================
# 4. ANÁLISE DE CLUSTER PELO MÉTODO DE WARD
# ============================================================

cluster_model = AgglomerativeClustering(
    n_clusters=2,
    linkage="ward",
)

cluster_labels = cluster_model.fit_predict(X)

df_pca["Cluster_num"] = cluster_labels


# ============================================================
# 5. NOMEAÇÃO DOS CLUSTERS
# ============================================================

# O grupo com maior média no Escore Geral é denominado
# "Florescimento". O outro é denominado "Desenvolvimento".

media_geral_por_cluster = (
    df_pca.groupby("Cluster_num")["Escore_Geral"]
    .mean()
)

cluster_florescimento = media_geral_por_cluster.idxmax()

df_pca["Cluster"] = df_pca["Cluster_num"].apply(
    lambda cluster: (
        "Florescimento"
        if cluster == cluster_florescimento
        else "Desenvolvimento"
    )
)


# ============================================================
# 6. ANÁLISE DE COMPONENTES PRINCIPAIS
# ============================================================

pca = PCA(n_components=2)

principal_components = pca.fit_transform(X)

df_pca["PC1"] = principal_components[:, 0]
df_pca["PC2"] = principal_components[:, 1]

variancia_explicada = pca.explained_variance_ratio_

pc1_percentual = variancia_explicada[0] * 100
pc2_percentual = variancia_explicada[1] * 100
variancia_acumulada = variancia_explicada.sum() * 100


# ============================================================
# 7. INFORMAÇÕES NO CONSOLE
# ============================================================

print("Variância explicada pela PCA:")
print(f"PC1: {pc1_percentual:.2f}%")
print(f"PC2: {pc2_percentual:.2f}%")
print(f"PC1 + PC2: {variancia_acumulada:.2f}%")

print("\nQuantidade de participantes por cluster:")
print(df_pca["Cluster"].value_counts())


# ============================================================
# 8. GRÁFICO DA PCA
# ============================================================

fig, ax = plt.subplots(figsize=(9, 7))

ordem_clusters = [
    "Florescimento",
    "Desenvolvimento",
]

for cluster_name in ordem_clusters:

    subset = df_pca[df_pca["Cluster"] == cluster_name]

    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        label=f"{cluster_name} (n={len(subset)})",
        alpha=0.8,
    )


# ============================================================
# 9. POSIÇÃO MÉDIA DOS CLUSTERS
# ============================================================

centroides = (
    df_pca.groupby("Cluster")[["PC1", "PC2"]]
    .mean()
)

for cluster_name, coordinates in centroides.iterrows():

    ax.text(
        coordinates["PC1"],
        coordinates["PC2"],
        cluster_name,
        fontsize=10,
        horizontalalignment="center",
        verticalalignment="center",
    )


# ============================================================
# 10. FORMATAÇÃO DO GRÁFICO
# ============================================================

ax.set_title(
    "PCA das cinco dimensões do PERMA por cluster"
)

ax.set_xlabel(
    f"PC1 ({pc1_percentual:.1f}% da variância)"
)

ax.set_ylabel(
    f"PC2 ({pc2_percentual:.1f}% da variância)"
)

ax.legend()

ax.grid(True)

plt.tight_layout()


# ============================================================
# 11. SALVAMENTO
# ============================================================

output_path = Path(OUTPUT_IMAGE)

plt.savefig(
    output_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(f"\nGráfico salvo em: {output_path.resolve()}")

ImportError: Missing optional dependency 'pyreadstat'.  Use pip or conda to install pyreadstat.